# Write ATST example 4

Read the source data, define each ATST block, and write one self-contained ATST file.

## Setup

Import the ATST block classes, writer, device reader, and data utilities.

In [1]:
from pathlib import Path

import pandas as pd

from ATST import ATSTFile, write_atst
from ATST.blocks import (
    Assay,
    Entities,
    EntityTable,
    FileMetadata,
    Layout,
    Metadata,
    Readings,
    Study,
)
from ATST import LogPhase600Reader


## Input data

Read the device export and layout. The reader supplies the measurements plus instrument metadata and assay information.

In [2]:
raw_path = Path("raw_outputs/simulated_logphase_output.txt")
device_output = LogPhase600Reader().read(raw_path.read_bytes(), filename=raw_path.name)
layout_data = pd.read_csv("example_4_layout.tsv", sep="\t", keep_default_na=False)

device_output.readings.head()


,Time,A1,A2,A3,A4,A5,A6,A7,A8,A9,...,H3,H4,H5,H6,H7,H8,H9,H10,H11,H12
0,0,0.168,0.196,0.171,0.140,0.188,0.163,0.177,0.170,0.151,...,0.186,0.186,0.169,0.168,0.170,0.165,0.154,0.107,0.110,0.098
1,2400,0.179,0.181,0.171,0.190,0.188,0.168,0.173,0.175,0.183,...,0.173,0.209,0.170,0.141,0.177,0.158,0.165,0.101,0.087,0.111
2,4800,0.217,0.235,0.233,0.209,0.211,0.211,0.215,0.226,0.233,...,0.184,0.193,0.171,0.151,0.210,0.203,0.219,0.099,0.103,0.102
3,7200,0.282,0.271,0.267,0.281,0.275,0.270,0.287,0.274,0.273,...,0.191,0.202,0.161,0.166,0.279,0.281,0.265,0.086,0.099,0.096
4,9600,0.340,0.352,0.310,0.371,0.369,0.346,0.349,0.349,0.352,...,0.189,0.188,0.175,0.153,0.349,0.357,0.344,0.092,0.108,0.101


## FILE_INFO

`FILE_INFO` identifies the output file and ATST format. The file name, format, version, creation date, delimiter, and encoding are required.

In [3]:
file_info = FileMetadata(data={
    "file_name": "example_4.atst.txt",
    "format": "ATST",
    "format_version": "0.1",
    "created_on": "2026-01-01",
    "field_delimiter": "TAB",
    "encoding": "UTF-8",
})


## STUDY

`STUDY` identifies and describes the experiment. A non-empty title and study ID are required.

In [4]:
study = Study(data={
    "title": "Example 4 simulated LogPhase600 plate",
    "study_id": "ATST_EXAMPLE_4",
    "description": "A complete in-file simulated 96-well OD600 assay.",
})


## METADATA

`METADATA` records how the measurement was acquired. The block is required, but it has no mandatory fields.

In [5]:
metadata = Metadata(data=device_output.metadata)


## ASSAY

`ASSAY` describes the measured signal. Readout type, readout unit, and time unit are required; a plate layout also requires plate format.

In [6]:
assay = Assay(data=device_output.assay)


## ENTITIES

`ENTITIES` defines the biological records referenced by the layout. Each table declares a primary key with unique, non-empty values.

In [7]:
phages = EntityTable(
    name="ENTITIES",
    table_name="PHAGES",
    pk="PHAGE_ID",
    data=pd.read_csv("entities/PHAGES.csv", keep_default_na=False),
)
isolates = EntityTable(
    name="ENTITIES",
    table_name="ISOLATES",
    pk="ISOLATE_ID",
    data=pd.read_csv("entities/ISOLATES.csv", keep_default_na=False),
)
entities = Entities(tables={"PHAGES": phages, "ISOLATES": isolates})


## LAYOUT

`LAYOUT` maps each readings column to its experimental and biological context. Its `well_loc` or `curve_id` key must be unique and match the readings columns.

In [8]:
layout = Layout(data=layout_data)


## READINGS

`READINGS` contains the time series. `Time` is the first column and the remaining column names must match layout keys.

In [9]:
readings = Readings(data=device_output.readings)


## Write the ATST file

Combine the blocks and write the complete single-file dataset.

In [10]:
atst = ATSTFile(
    file_info=file_info,
    study=study,
    metadata=metadata,
    assay=assay,
    entities=entities,
    layout=layout,
    readings=readings,
)
write_atst(atst, "example_4.atst.txt")


PosixPath('example_4.atst.txt')